# mvp_model: the LAST Kaggle run (everything that is left, in one notebook)
Run the cells top to bottom (or **Save Version -> Save & Run All (Commit)**, then close the page). Only cell 1 can be edited, normally not at all.
It trains, in this order, and writes ONE results zip:
1. **wedge** (H11): the leak check, 3 seeds x 5 folds (about 18 min)
2. **final models**: `full_hn` and `hist_hn` trained on ALL 95 environments, 3 seeds (about 10 min)
3. **transfer** (H10): `full_hn` and `hist_hn` with scenario B, C or D held out (about 25 min)
Total about 1 hour on a T4. Needs the NEW packed file (made with the new `pack_dataset.py`) and the NEW code. Guide: `docs/steps.md`, section M5e/M6.

In [ ]:
# ---- cell 1: settings (normally nothing to change) ----
RUN_NAME = 'final'
SEEDS    = '0,1,2'
EPOCHS   = 60
PATIENCE = 10
DO_WEDGE, DO_FINAL, DO_TRANSFER = True, True, True

In [ ]:
# ---- cell 2: find the code and the data, set up the working folder (Kaggle or Colab) ----
import glob, os, shutil, zipfile
ON_KAGGLE = os.path.exists('/kaggle/input')
WORK = '/kaggle/working' if ON_KAGGLE else '/content'
ROOTS = ['/kaggle/input'] if ON_KAGGLE else ['/content', '/content/drive/MyDrive']
def find(pattern):
    for r in ROOTS:
        hits = [h for h in glob.glob(f'{r}/**/{pattern}', recursive=True) if '/mvp_model/mvp_model/' not in h]
        if hits:
            return hits[0]
    return None
train_py = find('train.py')
if train_py is None:                                        # the code was uploaded as a zip that was not unpacked
    z = find('mvp_model*.zip')
    assert z, 'No train.py and no mvp_model zip found. Did you add the dataset as Input / upload the files?'
    zipfile.ZipFile(z).extractall(WORK + '/unzipped')
    train_py = glob.glob(WORK + '/unzipped/**/train.py', recursive=True)[0]
SRC = os.path.dirname(train_py)
CODE = WORK + '/mvp_model'
if os.path.abspath(SRC) != os.path.abspath(CODE):
    shutil.copytree(SRC, CODE, dirs_exist_ok=True, ignore=shutil.ignore_patterns('runs', '__pycache__', 'reports', 'packed'))
DATA = find('*_packed.npz')
assert DATA, 'mvp_v1_packed.npz not found. Upload it together with the code (see the guide).'
os.chdir(CODE)
print('platform:', 'Kaggle' if ON_KAGGLE else 'Colab / other')
print('code    :', CODE, '(from', SRC + ')')
print('data    :', DATA, f'({os.path.getsize(DATA) / 1e6:.1f} MB)')

In [ ]:
# ---- cell 3: is there a GPU? ----
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    x = torch.randn(1000, 1000, device='cuda'); print('GPU test ok:', float((x @ x).sum()) != 0)
else:
    print('NO GPU. Right panel -> Session options -> Accelerator -> choose a GPU, then Run All again.')

In [ ]:
# ---- cell 4: check the networks (must end with TORCH CHECK PASSED) ----
!python tests/test_torch.py

In [ ]:
# ---- cell 5: smoke test (about 2 minutes): every mode once, 1 epoch, written to runs_smoke (never mixed with real results) ----
!python train.py "{DATA}" --variant wedge --seeds 0 --epochs 1 --fold 0 --out runs_smoke
!python train.py "{DATA}" --variant full_hn --seeds 0 --epochs 1 --final --out runs_smoke
!python train.py "{DATA}" --variant full_hn --seeds 0 --epochs 1 --holdout B --out runs_smoke

In [ ]:
# ---- cell 6: the real training (the long cell). Finished parts are skipped, so after an interruption just run it again. ----
import time
os.makedirs('runs/logs', exist_ok=True)
def run(args, log):
    t0 = time.time()
    print('=' * 20, args, '=' * 20)
    get_ipython().system(f'python train.py "{DATA}" {args} --seeds {SEEDS} --epochs {EPOCHS} --patience {PATIENCE} 2>&1 | tee -a runs/logs/{log}.log')
    print(f'>>> finished in {(time.time() - t0) / 60:.1f} min')
if DO_WEDGE:
    run('--variant wedge', 'wedge')
if DO_FINAL:
    for v in ('full_hn', 'hist_hn'):
        run(f'--variant {v} --final', f'{v}_final')
if DO_TRANSFER:
    for v in ('full_hn', 'hist_hn'):
        run(f'--variant {v} --holdout B,C,D', f'{v}_holdout')

In [ ]:
# ---- cell 7: pack the results into ONE zip to download ----
import zipfile
zname = f'{WORK}/results_{RUN_NAME}.zip'
n = 0
with zipfile.ZipFile(zname, 'w', zipfile.ZIP_DEFLATED) as z:
    for pat in ('runs/oof_*.npz', 'runs/models/*.pt', 'runs/logs/*.log'):
        for f in sorted(glob.glob(pat)):
            z.write(f); n += 1
print(f'{n} files -> {zname} ({os.path.getsize(zname) / 1e6:.1f} MB)')
if ON_KAGGLE:
    print('Download it: notebook page -> Output tab -> results_final.zip   (after "Save Version" if you ran with Save & Run All)')
else:
    from google.colab import files
    files.download(zname)